# Sky Collector — Build Android APK

Загрузи файлы игры, нажми **Runtime → Run all**.
Через ~15 минут APK появится справа в файлах.

In [ ]:
# @title 1. Установка зависимостей
import os, sys, shutil, zipfile, urllib.request, warnings, time, glob, base64
from pathlib import Path

print("[*] Installing build tools...")
!apt-get update -qq && apt-get install -y -qq git zip unzip openjdk-17-jdk python3-pip autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev libltdl-dev ccache curl wget > /dev/null 2>&1
!pip install --upgrade pip setuptools wheel cython buildozer -q
print("[+] Dependencies installed")

In [ ]:
# @title 2. Загрузи файлы игры
from google.colab import files
import zipfile
import io

print("[!] Загрузи архив с игрой (.zip):")
print("    В архиве должны быть: main.py, buildozer.spec, requirements.txt")
print("    (можно также .github/workflows/build-apk.yml)")

uploaded = files.upload()

# Создаём рабочую папку
PROJECT = "/content/skycollector"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
os.makedirs(PROJECT, exist_ok=True)

for fname, content in uploaded.items():
    if fname.endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(content)) as zf:
            zf.extractall(PROJECT)
        print(f"[+] Extracted {fname}")
    else:
        dest = os.path.join(PROJECT, fname)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        with open(dest, "wb") as f:
            f.write(content)
        print(f"[+] Saved {fname}")

os.chdir(PROJECT)
print(f"\n[*] Working in {PROJECT}")
!ls -la

In [ ]:
# @title 3. Сборка APK
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

print("[*] Buildozer is starting... (this takes ~10-15 min on first run)")
print("[*] After first build, subsequent runs are faster (cached)")

start = time.time()
result = !buildozer android debug 2>&1
elapsed = (time.time() - start) / 60

print("\n" + "=" * 50)
print(f"Build finished in {elapsed:.1f} minutes")

# Show last 5 lines
lines = [l for l in result if l.strip()]
for l in lines[-5:]:
    print(l)

In [ ]:
# @title 4. Скачай APK
import glob
from google.colab import files

apks = glob.glob("/content/skycollector/bin/*.apk") + glob.glob("bin/*.apk")

if apks:
    apk_path = apks[0]
    size_mb = os.path.getsize(apk_path) / (1024 * 1024)
    print(f"[+] APK ready: {apk_path} ({size_mb:.1f} MB)")
    files.download(apk_path)
else:
    print("[!] APK not found. Check build logs above for errors.")
    print("    Possible fixes:")
    print("    - Run 'buildozer android debug' manually in the cell above")
    print("    - Check that main.py and buildozer.spec exist in the project root")
    !find /content -name "*.apk" 2>/dev/null

In [ ]:
# @title 5. Быстрый старт (если APK не нашёлся)
import os
os.chdir("/content/skycollector")
print("Project files:")
!find . -maxdepth 3 -not -path './.buildozer/*' | head -30
print("\n---")
print("Run manually if needed:")
print("  cd /content/skycollector && buildozer android debug")